# MRS plots
---
### author: Juju Fars
### date: 2024-12

Loads the refined MRS results (`Data/Refined/mrs_results_main*.xlsx`, produced by
`process_data.ipynb`) and plots pre- vs post-training GABA+ and Glx concentrations
in early visual cortex (V1/EVC) and sensorimotor cortex (M1/SMC). The second half of
the notebook ("ABC values") correlates the metabolite changes with behavioural
training-effect measures (ABC) from the psychophysics tasks.

In [ ]:
import importlib
import import_packages # all my packages are there, it's easier
importlib.reload(import_packages) # so I can add new packages in import_packages.py without restarting the kernel
from import_packages import *
print("Packages (re)loaded successfully!")

In [ ]:
# Set Up the paths 
ori = os.getcwd()
path_data_ref = os.path.abspath(os.path.join(ori, "..", "Data", "Refined"))
path_plot = os.path.abspath(os.path.join(ori, "..", "Data", "Plots"))

In [ ]:
mrs_results = pd.read_excel(path_data_ref+'/mrs_results_main.xlsx', sheet_name=None) 
mrs_results_new = pd.read_excel(path_data_ref+'/mrs_results_main_new.xlsx', sheet_name=None) 

In [ ]:
# functions 

def plot_grouped_scatter(
    df,
    group_by=['stage', 'amb', 'location', 'Metab'],
    cols=3,
    figsize=(15, 5),
    x='cutoff_frequency_cpd',
    y='metab_concentration',
    hue='ID',
    style='ID',
    palette='viridis',
    s=100,
    scatter=True,
    color='black',
    line_kws={'linestyle': 'dashed'},
    corr_test='pearson',
    ax_title_format='{stage} | {amb} | {location} | {metab}',
    xlabel='Cutoff Frequency (cpd)',
    ylabel='Metabolite Concentration',
    save_path=None
):


    # Group the DataFrame
    grouped = df.groupby(group_by)

    # Calculate the number of rows and columns for the subplots grid
    num_plots = len(grouped)
    rows = (num_plots // cols) + (num_plots % cols > 0)  # Automatically calculate rows

    # Create the figure and axes for subplots
    fig, axes = plt.subplots(rows, cols, figsize=(figsize[0], figsize[1] * rows))
    axes = axes.flatten()

    # Iterate through each group
    for idx, (group_keys, group_data) in enumerate(grouped):
        ax = axes[idx]

        # Unpack group keys into variables (filling missing values with 'NaN')
        group_dict = {key: val for key, val in zip(group_by, group_keys)}
        stage_val = group_dict.get('stage', 'NaN')
        amb_val = group_dict.get('amb', 'NaN')
        location_val = group_dict.get('location', 'NaN')
        metab_val = group_dict.get('Metab', 'NaN')

        # Scatter plot
        if scatter:
            sns.scatterplot(
                data=group_data,
                x=x,
                y=y,
                hue=hue,
                style=style,
                palette=palette,
                s=s,
                ax=ax
            )

        # Regression line
        sns.regplot(
            data=group_data,
            x=x,
            y=y,
            scatter=False,
            color=color,
            line_kws=line_kws,
            ax=ax
        )

        # Correlation test
        if corr_test == 'pearson' and len(group_data) > 1:
            r_value, p_value = pearsonr(group_data[x], group_data[y])
            ax.text(
                0.05, 0.95,
                f'R = {r_value:.2f}\nP = {p_value:.3f}',
                transform=ax.transAxes,
                ha='left', va='top', fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", edgecolor='gray', facecolor='white')
            )

        # Set plot titles and labels dynamically
        ax.set_title(ax_title_format.format(
            stage=stage_val, amb=amb_val, location=location_val, metab=metab_val
        ))
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

    # Hide unused axes
    for idx in range(num_plots, len(axes)):
        fig.delaxes(axes[idx])

    # Adjust layout
    plt.tight_layout()

    # Save or show the plot
    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()


# function for scatterplots

def plot_scatter(
    df,
    x='cutoff_diff',
    y='metab_diff',
    style='ID',
    palette='viridis',
    s=100,
    scatter=True,
    color='black',
    line_kws={'linestyle': 'dashed'},
    corr_test='pearson',
    xlabel='Cutoff Frequency Difference',
    ylabel='Metabolite Concentration Difference',
    save_path=None
):

    # Create the figure and axes for subplots
    fig, ax = plt.subplots(figsize=(5, 3))

    # Scatter plot
    if scatter:
        scatter_plot = sns.scatterplot(
            data=df,
            x=x,
            y=y,
            style=style,
            palette=palette,
            s=s,
            ax=ax
        )

    # Regression line
    sns.regplot(
        data=df,
        x=x,
        y=y,
        scatter=False,
        color=color,
        line_kws=line_kws,
        ax=ax
    )

    # Correlation test
    if corr_test == 'pearson' and len(df) > 1:
        r_value, p_value = pearsonr(df[x], df[y])
        ax.text(
            0.05, 0.95,
            f'R = {r_value:.2f}\nP = {p_value:.3f}',
            transform=ax.transAxes,
            ha='left', va='top', fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='gray', facecolor='white')
        )

    # Set plot titles and labels
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Adjust layout
    plt.tight_layout()

    # Move the legend outside the plot
    scatter_plot.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='Legend')

    # Save or show the plot
    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()

In [ ]:
# Melt the data to make it suitable for seaborn (long format)
df_melted = mrs_results['metab'].melt(id_vars=['ID', 'location', 'Metab'], value_vars=['pre', 'post'], 
                    var_name='Condition', value_name='Value')
df_melted['Condition'] = pd.Categorical(df_melted['Condition'], categories=['pre', 'post'])

# Create a new column that combines 'location' and 'Condition' for ordering
df_melted['location_condition'] = df_melted['location'] + " " + df_melted['Condition'].astype(str)
indexID = df_melted[ (df_melted['ID'] == 1) ].index # | (df_melted['ID'] == 2) | (df_melted['ID'] == 3)
df_melted.drop(indexID , inplace=True)

print(df_melted)

In [ ]:
df_melted_Gaba = df_melted.copy()
df_melted_Gaba = df_melted_Gaba[df_melted_Gaba['Metab'] == 'GABA+']
df_melted_Gaba['location'] = pd.Categorical(df_melted_Gaba['location'], categories=['V1', 'M1'], ordered=True)


df_melted_Glx = df_melted.copy()
df_melted_Glx = df_melted_Glx[df_melted_Glx['Metab'] == 'Glx']
df_melted_Glx['location'] = pd.Categorical(df_melted_Glx['location'], categories=['V1', 'M1'], ordered=True)


In [ ]:
df_melted_Gaba_pre = df_melted_Gaba[df_melted_Gaba['Condition'] == 'pre']
df_melted_Gaba_post = df_melted_Gaba[df_melted_Gaba['Condition'] == 'post']

In [ ]:
# Melt the data to make it suitable for seaborn (long format)
df_melted = mrs_results_new['metab'].melt(id_vars=['ID', 'location', 'Metab'], value_vars=['pre', 'post'], 
                    var_name='Condition', value_name='Value')
df_melted['Condition'] = pd.Categorical(df_melted['Condition'], categories=['pre', 'post'])

# Create a new column that combines 'location' and 'Condition' for ordering
df_melted['location_condition'] = df_melted['location'] + " " + df_melted['Condition'].astype(str)
indexID = df_melted[ (df_melted['ID'] == 1) ].index # | (df_melted['ID'] == 2) | (df_melted['ID'] == 3)
df_melted.drop(indexID , inplace=True)

df_melted_Gaba_new = df_melted.copy()
df_melted_Gaba_new = df_melted_Gaba_new[df_melted_Gaba_new['Metab'] == 'GABA+']
df_melted_Gaba_new['location'] = pd.Categorical(df_melted_Gaba_new['location'], categories=['V1', 'M1'], ordered=True)

df_melted_Glx_new = df_melted.copy()
df_melted_Glx_new = df_melted_Glx_new[df_melted_Glx_new['Metab'] == 'Glx']
df_melted_Glx_new['location'] = pd.Categorical(df_melted_Glx_new['location'], categories=['V1', 'M1'], ordered=True)


In [ ]:
# check the correlation 

corr = pg.corr(
    y=df_melted_Glx['Value'].values,
    x=df_melted_Glx_new['Value'].values
)

print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    y=df_melted_Glx['Value'],
    x=df_melted_Glx_new['Value'],
    alpha=0.7,
    ax=ax
)

ax.set_ylabel("Old fsl_mrs package values - Glx", loc = 'center')
ax.set_xlabel("New fsl_mrs package values - Glx")

# ax.text(
#     0.05, 0.95,
#     f"N = 20, R = 0.067\np = 0.778, CI95% = [-0.39, 0.5]",
#     transform=ax.transAxes,
#     ha="left",
#     va="top"
# )
#plt.savefig('../Data/Plots/Narrative_plots/v3/Corr_GABA_CUTOFF.png', dpi=600, bbox_inches='tight')

plt.show()

In [ ]:
# extra - correlation between Gaba+ and Glx

# Pivot the DataFrame to get GABA+ and Glx in separate columns
pivot_df = df_melted_Gaba_post.pivot(index=['ID'], columns='location', values='Value').reset_index()

indexID = pivot_df[ (pivot_df['ID'] == 1)].index
pivot_df.drop(indexID , inplace=True)
print(pivot_df)


In [ ]:
plot_id = mrs_results['metab'].copy()
indexID = plot_id[ (plot_id['ID'] == 1) | (plot_id['ID'] == 2) | (plot_id['ID'] == 3)].index # | (plot_id['ID'] == 2) | (plot_id['ID'] == 3)
plot_id.drop(indexID , inplace=True)
plot_id = plot_id[(plot_id['Metab'] == 'GABA+')]
plot_id = plot_id[(plot_id['location'] == 'V1')]

In [ ]:
print(df_melted_Gaba)

In [ ]:


# Set a better style
sns.set_theme(style='white', palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each location condition
ax = sns.boxplot(
    x='location', y='Value', data=df_melted_Gaba, hue='Condition', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'post': "#009E73", 'pre': "#D55E00"}
)
# palette={'post': '#009E73', 'pre': '#D55E00'}

strip = sns.stripplot(data=df_melted_Gaba, x="location", y="Value", hue='Condition', dodge=True, ax=ax, 
                palette={'post': '#009E73', 'pre': '#D55E00'}, edgecolor='black', linewidth=1)

# ax.set_yticks(range(0, 4, 0))
#ax.set_yticklabels([5, 10, 15, 20, 25])
ax.set_ylim(ymin=1.5, ymax=3)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1])],
          labels=['Pre-training', 'Post-training'],
          loc='upper right', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})



# Customize axis labels
ax.set_ylabel("GABA+ concentration (mM/kg)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

customx_labels = ['EVC', 'SMC']
ax.set_xticks(np.arange(len(customx_labels)))
ax.set_xticklabels(customx_labels, fontsize=14)
# Get cleaned categorical order for location

location_order = df_melted_Gaba['location'].cat.categories
location_mapping = {val: i for i, val in enumerate(location_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'location' then 'Condition'
df_sorted = df_melted_Gaba.sort_values(by=['location', 'Condition'], key=lambda col: col.astype(str))

# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, location) and Condition
coord_map = {}  # {(ID, location): {Condition: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['location'])
    Condition = row['Condition']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][Condition] = (x, y)

# Plot lines for each pair of Pre/Post
for key, Conditions in coord_map.items():
    if 'pre' in Conditions and 'post' in Conditions:
        x_vals = [Conditions['pre'][0], Conditions['post'][0]]
        y_vals = [Conditions['pre'][1], Conditions['post'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 2.8, 0.025, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)
h2 = 2.8
ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h2-0.25, h2+h-0.25, h2+h-0.25, h2-0.25], lw=1.5, c=col)
ax.text((x2+x2)*.5, h2+h+h-0.25, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x1+x1)*.5, h2+h+h, "*", ha='center', va='bottom', color=col)

# plt.title('GABA+ Pre- and Post-training')
# plt.figure(figsize=(8, 6))

# plt.savefig('../Data/Plots/Narrative_plots/v3/GABA_EVC_SMC.png', dpi=600, bbox_inches='tight')



####################################################################################




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

sns.set_theme(style='white', palette="pastel")
palette = {'post': "#009E73", 'pre': "#D55E00"}

# Mapping between actual df codes and friendly labels
location_name_map = {'V1': 'EVC', 'M1': 'SMC'}
# reverse map so user can pass 'EVC' or 'SMC' and we find the code
friendly_to_code = {friendly: code for code, friendly in location_name_map.items()}

def plot_gaba_location_with_mapping(df, requested_label, save_path=None, show_diag=True):
    """
    df: dataframe with real location codes in df['location'] (e.g. 'V1' and 'M1')
    requested_label: either the real code ('V1'/'M1') or friendly label ('EVC'/'SMC')
    """
    # Normalize requested_label input
    req = str(requested_label).strip()
    # If user passed friendly label, convert to real code
    if req in friendly_to_code:
        code_label = friendly_to_code[req]
        friendly_label = req
    elif req in location_name_map:
        code_label = req
        friendly_label = location_name_map.get(req, req)
    else:
        # Try tolerant matching against both sets (case-insensitive / substring)
        all_codes = list(location_name_map.keys())
        all_friendly = list(location_name_map.values())
        # try case-insensitive exact match to code
        matches_code = [c for c in all_codes if c.lower() == req.lower()]
        matches_friendly = [f for f in all_friendly if f.lower() == req.lower()]
        if matches_code:
            code_label = matches_code[0]
            friendly_label = location_name_map.get(code_label, code_label)
        elif matches_friendly:
            friendly_label = matches_friendly[0]
            code_label = friendly_to_code[friendly_label]
        else:
            # substring fuzzy fallback
            matches_code = [c for c in all_codes if req.lower() in c.lower()]
            matches_friendly = [f for f in all_friendly if req.lower() in f.lower()]
            if matches_code:
                code_label = matches_code[0]
                friendly_label = location_name_map.get(code_label, code_label)
            elif matches_friendly:
                friendly_label = matches_friendly[0]
                code_label = friendly_to_code[friendly_label]
            else:
                # nothing matched — report and exit
                print(f"Requested label {requested_label!r} not recognized.")
                print("Known codes:", all_codes)
                print("Known friendly labels:", all_friendly)
                return

    # Diagnostics: what values exist
    all_locs = sorted(df['location'].astype(str).unique())
    if show_diag:
        print(f"All distinct location values in df: {all_locs}")
        print(f"Resolved request {requested_label!r} -> code '{code_label}' (friendly '{friendly_label}').")

    # Filter using the actual df code (e.g. 'V1' or 'M1')
    df_loc = df[df['location'].astype(str) == code_label].copy()
    if df_loc.shape[0] == 0:
        # try tolerant alternatives just in case (case-insensitive, substring)
        df_loc = df[df['location'].astype(str).str.lower() == code_label.lower()].copy()
    if df_loc.shape[0] == 0:
        df_loc = df[df['location'].astype(str).str.lower().str.contains(code_label.lower())].copy()

    if df_loc.shape[0] == 0:
        print(f"WARNING: no rows found for code '{code_label}'. Aborting plot.")
        return

    # Standardize Condition column to lowercase 'pre'/'post' if possible
    df_loc['Condition_orig'] = df_loc['Condition'].astype(str)
    df_loc['Condition'] = df_loc['Condition_orig'].str.lower().str.strip()

    # Attempt to map common variants of pre/post to our keys
    cond_unique = df_loc['Condition'].unique().tolist()
    mapping = {}
    if not set(cond_unique).issubset(set(palette.keys())):
        if any(c.lower().startswith('pre') for c in cond_unique):
            mapping.update({c: 'pre' for c in cond_unique if c.lower().startswith('pre')})
        if any(c.lower().startswith('post') for c in cond_unique):
            mapping.update({c: 'post' for c in cond_unique if c.lower().startswith('post')})
    if mapping:
        df_loc['Condition'] = df_loc['Condition'].map(lambda c: mapping.get(c, c))

    # Set explicit order for plotting
    condition_order = [c for c in ['pre', 'post'] if c in df_loc['Condition'].unique()]

    # Create independent figure
    plt.figure(figsize=(3, 6))
    ax = plt.gca()

    # Boxplot
    sns.boxplot(
        x='Condition', y='Value', data=df_loc,
        showfliers=False, width=0.6, linewidth=1.5,
        palette=palette, ax=ax, order=condition_order
    )

    # Stripplot (no hue to avoid duplicated legends)
    strip = sns.stripplot(
        data=df_loc, x='Condition', y='Value',
        dodge=False, jitter=0.15,
        palette=palette, edgecolor='black', linewidth=1,
        ax=ax, order=condition_order
    )

    # Remove auto legend if created
    if getattr(ax, "legend_", None) is not None:
        ax.legend_.remove()

    # Axis tweaks
    ax.set_ylim(1.5, 3)
    ax.set_ylabel("GABA+ concentration (mM/kg)", fontsize=14)
    ax.set_xlabel("")
    ax.set_title(friendly_label, fontsize=16)
    # Thicker tick marks
    ax.tick_params(axis='both', width=2, length=6)

    # Thicker box (spines)
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    # Extract offsets for the stripplot and draw pre->post lines per subject
    offsets = []
    for coll in strip.collections:
        try:
            offsets.extend(coll.get_offsets())
        except Exception:
            pass

    df_sorted = df_loc.copy().sort_values(by=['Condition', 'ID'], key=lambda col: col.astype(str)).reset_index(drop=True)
    coord_map = {}
    for i, (_, row) in enumerate(df_sorted.iterrows()):
        if i >= len(offsets):
            break
        x, y = offsets[i]
        coord_map.setdefault(row['ID'], {})[row['Condition']] = (x, y)

    n_lines = 0
    for ID, conds in coord_map.items():
        if 'pre' in conds and 'post' in conds:
            ax.plot([conds['pre'][0], conds['post'][0]],
                    [conds['pre'][1], conds['post'][1]],
                    color='black', alpha=0.4, linewidth=1)
            n_lines += 1

    if show_diag:
        print(f"Filtered rows: {df_loc.shape[0]}; plotted {n_lines} subject lines.")

    # Manual legend
    handles = []
    if 'pre' in df_loc['Condition'].unique():
        handles.append(Line2D([0], [0], marker='s', color='w', markerfacecolor=palette['pre'], markersize=10, label='Pre-training'))
    if 'post' in df_loc['Condition'].unique():
        handles.append(Line2D([0], [0], marker='s', color='w', markerfacecolor=palette['post'], markersize=10, label='Post-training'))
    if handles:
        ax.legend(handles=handles, loc='upper right')

    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches='tight')

    plt.show()


# Example usage — two separate independent plots:
plot_gaba_location_with_mapping(
    df_melted_Gaba,
    requested_label='EVC',   # or 'V1'
    #save_path='../Data/Plots/Narrative_plots/v3/GABA_EVC.png'
)

plot_gaba_location_with_mapping(
    df_melted_Gaba,
    requested_label='SMC',   # or 'M1'
    #save_path='../Data/Plots/Narrative_plots/v3/GABA_SMC.png'
)


In [ ]:
location_name_map = {'V1': 'EVC', 'M1': 'SMC'}

ax = mrs.plot_paired_sessions(
    df_melted_Glx, requested_label='EVC', y_col='Value', y_label='Glx concentration (mM/kg)',
    session_col='Condition', id_col='ID', location_name_map=location_name_map, ylim=(3, 7)
)
ax.text(0.74, 5.25, "n.s.", ha='center', va='bottom', color='black', fontsize=18)
# save_path='../Data/Plots/new_new/Glx_SMC.png'
plt.show()


In [ ]:

offsets = {
    4: (0.05, -0.05),
    16: (0, -0.1), 
    20: (0.1, -0.1),
    21: (0, 0.1),
    14: (-0.05, -0.05),
    17: (0.1, -0.1), 
    12: (-0.05, 0.05)
}

fig, ax = plt.subplots(figsize=(4, 4))

for _, row in plot_id.iterrows():
    x = row['pre']
    y = row['post']
    id_ = int(row['ID'])

    if id_ in offsets:
        dx, dy = offsets[id_]

        # draw a small dot at the true data point only for shifted labels
        ax.plot(x, y, 'ko', markersize=3, zorder=3)

        ha = 'left' if dx > 0 else ('right' if dx < 0 else 'center')
        va = 'bottom' if dy > 0 else ('top' if dy < 0 else 'center')

        ax.annotate(
            str(id_),
            xy=(x, y),
            xytext=(x + dx, y + dy),
            fontsize=14,
            ha=ha, va=va,
            xycoords='data',
            textcoords='data',
            arrowprops=dict(
                arrowstyle='-',
                linewidth=1.0,
                color='black',
                shrinkA=0.1,   # shortens line so it doesn't go under the text
                shrinkB=0
            ),
            zorder=4
        )
    else:
        # regular labels: no dot, no connector
        ax.text(x, y, str(id_), fontsize=14, ha='center', va='center', zorder=4)

# reference line y = x
xmin = plot_id['pre'].min() - 2
xmax = plot_id['pre'].max() + 3
ax.plot([xmin, xmax], [xmin, xmax], '--', color='gray')

ax.set_xlim(1.7, 2.8)
ax.set_ylim(1.7, 2.8)
# ax.set_xticks(range(0, 6, 1))
# ax.set_yticks(range(0, 6, 1))
ax.set_xlabel("Pre-training GABA+ (mM/kg)")
ax.set_ylabel("Post-training GABA+ (mM/kg)")
ax.set_title("GABA+ Concentration (pre- vs post- training)")
ax.grid(True)

# plt.tight_layout()
plt.savefig('../Data/Plots/Narrative_plots/GABA_EVC_pre_vs_post.png', dpi=600, bbox_inches='tight')

# plt.show()

In [ ]:
# extra - for the prez

df_melted_Gaba_V1 = df_melted_Gaba[df_melted_Gaba['location'] == 'V1'].copy()

df_melted_Gaba_V1['location'] = df_melted_Gaba_V1['location'].cat.rename_categories(
    {'V1': 'Early Visual Cortex'}
)
df_melted_Gaba_V1['location'] = df_melted_Gaba_V1['location'].replace({'V1': 'Early Visual Cortex'})


In [ ]:
import matplotlib.patches as mpatches
# -------------------------
# 0) Sanity: example column names expected:
# df_melted_Gaba columns: ['ID', 'location', 'Metab', 'Condition', 'Value', ...]
# Condition values expected: 'pre' and 'post'
# -------------------------

# -------------------------
# 1) Filter to V1 and rename location label
# -------------------------
df_v1 = df_melted_Gaba[df_melted_Gaba['location'] == 'V1'].copy()

# If 'location' is categorical, rename the category; otherwise replace strings
try:
    if pd.api.types.is_categorical_dtype(df_v1['location']):
        df_v1['location'] = df_v1['location'].cat.rename_categories({'V1': 'Early Visual Cortex'})
    else:
        df_v1['location'] = df_v1['location'].replace({'V1': 'Early Visual Cortex'})
except Exception:
    # fallback
    df_v1['location'] = df_v1['location'].replace({'V1': 'Early Visual Cortex'})

df_v1['location'] = df_v1['location'].cat.remove_unused_categories()

# -------------------------
# 2) Plotting style & palette
# -------------------------
sns.set_theme(style="white")
palette_cond = {'pre': '#D55E00', 'post': '#009E73'}

fig, ax = plt.subplots(figsize=(3, 6))

# -------------------------
# 3) Boxplot (no fliers)
# -------------------------
# Note: x='location' will be a single category 'Early Visual Cortex' after filtering
sns.boxplot(
    x='location',
    y='Value',
    hue='Condition',
    data=df_v1,
    palette=palette_cond,
    showfliers=False,
    width=0.6,
    linewidth=1.25,
    ax=ax,
    zorder=1
)

# -------------------------
# 4) Deterministic jitter + x positions for each Condition
#    Because we have only one x-category, we place 'pre' and 'post' slightly left/right of center
# -------------------------
center_x = 0.0  # seaborn plots the single category at x=0
cond_base_offsets = {'pre': -0.18, 'post': 0.18}   # tweak to better align under box halves if desired

rng = np.random.default_rng(42)
unique_ids = df_v1['ID'].unique()
subject_jitter = {sid: float(rng.uniform(-0.08, 0.08)) for sid in unique_ids}
# single jitter per subject so the two points vertically align

# prepare the plotting DataFrame
df_pts = df_v1.copy()
df_pts['x_base'] = df_pts['Condition'].map(cond_base_offsets)
df_pts['x_plot'] = df_pts.apply(lambda r: center_x + r['x_base'] + subject_jitter[r['ID']], axis=1)
df_pts['color'] = df_pts['Condition'].map(palette_cond)

# -------------------------
# 5) Plot the points (filled with black edge)
# -------------------------
ax.scatter(
    df_pts['x_plot'],
    df_pts['Value'],
    c=df_pts['color'].tolist(),
    edgecolors='black',
    linewidths=0.8,
    s=56,
    alpha=0.95,
    zorder=3
)

# -------------------------
# 6) Draw connecting lines (use same jittered x_plot so they align)
# -------------------------
for sid, g in df_pts.groupby('ID'):
    # require both pre and post present
    if {'pre', 'post'} <= set(g['Condition']):
        pre = g[g['Condition'] == 'pre'].iloc[0]
        post = g[g['Condition'] == 'post'].iloc[0]
        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['Value'], post['Value']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=2
        )

# -------------------------
# 7) Legend: build a simple legend for pre/post (avoid duplicate handles from seaborn)
# -------------------------
pre_patch = mpatches.Patch(color=palette_cond['pre'], label='Pre-training')
post_patch = mpatches.Patch(color=palette_cond['post'], label='Post-training')
ax.legend(handles=[pre_patch, post_patch], loc='lower left', frameon=True)


x1, x2 = 0, 0  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 2.7, 0.025, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)
h2 = 2.75
#ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2-0.25, h2+h-0.25, h2+h-0.25, h2-0.25], lw=1.5, c=col)
#ax.text((x1+x1)*.5, h2+h+h-0.25, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x2+x2)*.5, h2+h+h, "*", ha='center', va='bottom', color=col)

# -------------------------
# 8) Formatting
# -------------------------
ax.set_xlabel('')
ax.set_ylabel('GABA+ concentration (mMol/kg)', fontsize=14)
ax.set_title('Early Visual Cortex', fontsize = 15)#('GABA+ Concentration is\nLower after Training', fontsize=17)
ax.set_ylim(1.5, 3.0)     # tweak as you need
# ax.set_xticks([0])
plt.xticks([-0.2, 0.2], ['Pre-Training', 'Post-Training'], fontsize = 13)  # matches 'order=' above

# ax.set_xticklabels(['Early Visual Cortex'], fontsize=14)
# ax.grid(axis='y', linestyle='--', alpha=0.5)


plt.tight_layout()
# plt.savefig('../Data/Plots/Narrative_plots/v3/GABA_EVC.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.patches as mpatches
# -------------------------
# 0) Sanity: example column names expected:
# df_melted_Gaba columns: ['ID', 'location', 'Metab', 'Condition', 'Value', ...]
# Condition values expected: 'pre' and 'post'
# -------------------------

# -------------------------
# 1) Filter to V1 and rename location label
# -------------------------
df_v1 = df_melted_Gaba[df_melted_Gaba['location'] == 'M1'].copy()

# If 'location' is categorical, rename the category; otherwise replace strings
try:
    if pd.api.types.is_categorical_dtype(df_v1['location']):
        df_v1['location'] = df_v1['location'].cat.rename_categories({'M1': 'Sensorimotor Cortex'})
    else:
        df_v1['location'] = df_v1['location'].replace({'M1': 'Sensorimotor Cortex'})
except Exception:
    # fallback
    df_v1['location'] = df_v1['location'].replace({'M1': 'Sensorimotor Cortex'})

df_v1['location'] = df_v1['location'].cat.remove_unused_categories()

# -------------------------
# 2) Plotting style & palette
# -------------------------
sns.set_theme(style="white")
palette_cond = {'pre': '#D55E00', 'post': '#009E73'}

fig, ax = plt.subplots(figsize=(3, 6))

# -------------------------
# 3) Boxplot (no fliers)
# -------------------------
# Note: x='location' will be a single category 'Early Visual Cortex' after filtering
sns.boxplot(
    x='location',
    y='Value',
    hue='Condition',
    data=df_v1,
    palette=palette_cond,
    showfliers=False,
    width=0.6,
    linewidth=1.25,
    ax=ax,
    zorder=1
)

# -------------------------
# 4) Deterministic jitter + x positions for each Condition
#    Because we have only one x-category, we place 'pre' and 'post' slightly left/right of center
# -------------------------
center_x = 0.0  # seaborn plots the single category at x=0
cond_base_offsets = {'pre': -0.18, 'post': 0.18}   # tweak to better align under box halves if desired

rng = np.random.default_rng(42)
unique_ids = df_v1['ID'].unique()
subject_jitter = {sid: float(rng.uniform(-0.08, 0.08)) for sid in unique_ids}
# single jitter per subject so the two points vertically align

# prepare the plotting DataFrame
df_pts = df_v1.copy()
df_pts['x_base'] = df_pts['Condition'].map(cond_base_offsets)
df_pts['x_plot'] = df_pts.apply(lambda r: center_x + r['x_base'] + subject_jitter[r['ID']], axis=1)
df_pts['color'] = df_pts['Condition'].map(palette_cond)

# -------------------------
# 5) Plot the points (filled with black edge)
# -------------------------
ax.scatter(
    df_pts['x_plot'],
    df_pts['Value'],
    c=df_pts['color'].tolist(),
    edgecolors='black',
    linewidths=0.8,
    s=56,
    alpha=0.95,
    zorder=3
)

# -------------------------
# 6) Draw connecting lines (use same jittered x_plot so they align)
# -------------------------
for sid, g in df_pts.groupby('ID'):
    # require both pre and post present
    if {'pre', 'post'} <= set(g['Condition']):
        pre = g[g['Condition'] == 'pre'].iloc[0]
        post = g[g['Condition'] == 'post'].iloc[0]
        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['Value'], post['Value']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=2
        )

# -------------------------
# 7) Legend: build a simple legend for pre/post (avoid duplicate handles from seaborn)
# -------------------------
pre_patch = mpatches.Patch(color=palette_cond['pre'], label='Pre-training')
post_patch = mpatches.Patch(color=palette_cond['post'], label='Post-training')
ax.legend(handles=[pre_patch, post_patch], loc='lower left', frameon=True)


x1, x2 = 0, 0  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 2.5, 0.025, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)
h2 = 2.55
#ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2-0.25, h2+h-0.25, h2+h-0.25, h2-0.25], lw=1.5, c=col)
#ax.text((x1+x1)*.5, h2+h+h-0.25, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x2+x2)*.5, h2+h+h, "n.s.", ha='center', va='bottom', color=col)

# -------------------------
# 8) Formatting
# -------------------------
ax.set_xlabel('')
ax.set_ylabel('GABA+ concentration (mMol/kg)', fontsize=14)
ax.set_title(None)#('GABA+ Concentration is\nLower after Training', fontsize=17)
ax.set_ylim(1.3, 2.8)     # tweak as you need
ax.set_xticks([0])
ax.set_xticklabels(['Sensorimotor Cortex'], fontsize=14)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../Data/Plots/Narrative_plots/v3/GABA_SMC.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import pingouin as pg 

ttest = pg.ttest(
    paired=True,
    x=plot_id['post'],
    y=plot_id['pre'] 
)

print(ttest.round(3))
print(plot_id['pre'].mean())
print(plot_id['pre'].std())

In [ ]:
id_M1 = mrs_results['metab'].copy()
indexID = id_M1[ (id_M1['ID'] == 1) | (id_M1['ID'] == 2) | (id_M1['ID'] == 3)].index
id_M1.drop(indexID , inplace=True)
id_M1 = id_M1[(id_M1['Metab'] == 'GABA+')]
id_M1 = id_M1[(id_M1['location'] == 'M1')]

ttest = pg.ttest(
    paired=True,
    x=id_M1['post'],
    y=id_M1['pre'] 
)

print(ttest.round(3))
print(id_M1['post'].mean())
print(id_M1['post'].std())

In [ ]:

# Set a better style
sns.set_theme(style='white', palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each location condition
ax = sns.boxplot(
    x='location', y='Value', data=df_melted_Glx, hue='Condition', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'post': "#00619E", 'pre': "#D50019"}
)

strip = sns.stripplot(data=df_melted_Glx, x="location", y="Value", hue='Condition', dodge=True, ax=ax, 
                palette={'post': "#00619E", 'pre': "#D50019"}, edgecolor='black', linewidth=1)

# ax.set_yticks(range(0, 4, 0))
#ax.set_yticklabels([5, 10, 15, 20, 25])
# ax.set_ylim(ymin=1.5, ymax=3)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1])],
          labels=['Pre-training', 'Post-training'],
          loc='lower left', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})



# Customize axis labels
ax.set_ylabel("Glx+ concentration (mM/kg)", fontsize=14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

customx_labels = ['EVC', 'SMC']
ax.set_xticks(np.arange(len(customx_labels)))
ax.set_xticklabels(customx_labels, fontsize=14)
# Get cleaned categorical order for location

location_order = df_melted_Glx['location'].cat.categories
location_mapping = {val: i for i, val in enumerate(location_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'location' then 'Condition'
df_sorted = df_melted_Glx.sort_values(by=['location', 'Condition'], key=lambda col: col.astype(str))

# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, location) and Condition
coord_map = {}  # {(ID, location): {Condition: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['location'])
    Condition = row['Condition']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][Condition] = (x, y)

# Plot lines for each pair of Pre/Post
for key, Conditions in coord_map.items():
    if 'pre' in Conditions and 'post' in Conditions:
        x_vals = [Conditions['pre'][0], Conditions['post'][0]]
        y_vals = [Conditions['pre'][1], Conditions['post'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 7.8, 0.025, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)
h2 = 7.8
h1 = 7
ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h1-0.25, h1+h-0.25, h1+h-0.25, h1-0.25], lw=1.5, c=col)
ax.text((x2+x2)*.5, h1+h+h-0.25, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x1+x1)*.5, h2+h+h, "n.s.", ha='center', va='bottom', color=col)

# plt.title('GABA+ Pre- and Post-training')
# plt.figure(figsize=(8, 6))
plt.show()
# plt.savefig('../Data/Plots/Narrative_plots/GABA_EVC_SMC.png', dpi=600, bbox_inches='tight')



####################################################################################




In [ ]:
id_glx = mrs_results['metab'].copy()
indexID = id_glx[ (id_glx['ID'] == 1) ].index#| (id_glx['ID'] == 2) | (id_glx['ID'] == 3)].index
id_glx.drop(indexID , inplace=True)
id_glx = id_glx[(id_glx['Metab'] == 'Glx')]
# id_glx = id_glx[(id_glx['location'] == 'V1')]
id_glx = id_glx[(id_glx['location'] == 'M1')]

ttest = pg.ttest(
    paired=True,
    x=id_glx['post'],
    y=id_glx['pre'] 
)

print(ttest.round(3))
print(id_glx['pre'].mean())
print(id_glx['pre'].std())

## MRS stats - corrections 

In [ ]:
# add corrections from Rideaux

# ABC values - load the data
Correlate pre/post metabolite concentration changes with the behavioural training-effect
(ABC) measures from `process_data.ipynb`, split by contrast-sensitivity and
interocular-imbalance tasks.

In [ ]:
corr_data_cs = pd.read_excel(path_data_ref+'/bv_4_results_ABC_diff.xlsx', sheet_name='abc_df_in') 
corr_data_bi = pd.read_excel(path_data_ref+'/bv_4_results_ABC_diff.xlsx', sheet_name='abc_df_bi') 
MRS_data = pd.read_excel(path_data_ref+'/mrs_results_main.xlsx') 

In [ ]:
corr_data = corr_data[corr_data['stage'] != 'familiarisation']
#print(corr_data)

MRS_data = MRS_data.drop(columns=['diff'])
MRS_data = MRS_data.melt(
    id_vars=["ID", "location", "Metab"], 
    value_vars=["pre", "post"], 
    var_name="stage", 
    value_name="value"
)
#print(MRS_data)

# Merge the DataFrames
merged_df_cs = pd.merge(
    corr_data_cs,
    MRS_data,
    left_on=["ID"],
    right_on=["ID"],
    how="inner"
)
merged_df_cs = merged_df_cs.rename(columns={"value": "metab_concentration"})

# Merge the DataFrames
merged_df_bi = pd.merge(
    corr_data_bi,
    MRS_data,
    left_on=["ID"],
    right_on=["ID"],
    how="inner"
)
merged_df_bi = merged_df_bi.rename(columns={"value": "metab_concentration"})

In [ ]:
merged_df_cs = merged_df_cs[(merged_df_cs['ID']!=1)&(merged_df_cs['ID']!=2)&(merged_df_cs['ID']!=3)]
merged_df_bi = merged_df_bi[(merged_df_bi['ID']!=1)&(merged_df_bi['ID']!=2)&(merged_df_bi['ID']!=3)]


# Contrast Sensitivity
Correlation between metabolite change and the contrast-sensitivity ABC measure.

In [ ]:
# Split the dataframe into pretraining and posttraining
pretraining = merged_df_cs[merged_df_cs['stage'] == 'pre'].copy()
posttraining = merged_df_cs[merged_df_cs['stage'] == 'post'].copy()

# Merge on common columns
merged = pd.merge(
    posttraining, 
    pretraining, 
    on=['ID', 'amb', 'location', 'ABC', 'Metab'], 
    suffixes=('_post', '_pre')
)

# Calculate the differences
merged['metab_difference'] = merged['metab_concentration_post'] - merged['metab_concentration_pre']

# Create a reduced DataFrame with only relevant columns
merged_df_cs_diff = merged[['ID', 'amb', 'location', 'Metab', 'metab_difference', 'ABC']]

# Display the resulting DataFrame
#print(merged_cutoff_diff)


plot_grouped_scatter(
    df=merged_df_cs_diff,
    group_by=['amb', 'location', 'Metab'],  # Grouping by 'amb' and 'location'
    cols=2,  # Number of columns in the subplot grid
    figsize=(15, 5),  # Base figure size
    x='ABC',  # X-axis column
    y='metab_difference',  # Y-axis column
    hue='ID',  # Hue for scatter plot
    style='ID',  # Style for scatter plot
    ax_title_format='{amb} | {location} | {metab}',  # Dynamic title based on grouping keys
    xlabel='Area Below Curve',  # X-axis label
    ylabel='Metabolite Concentration Difference (post-pre)',  # Y-axis label
    save_path=None  # Show the plot instead of saving
)
#############################################################################################################
#############################################################################################################





In [ ]:
merged_df_cs_diff_BV1G = merged_df_cs_diff[(merged_df_cs_diff['location']!='M1')]
merged_df_cs_diff_BV1G = merged_df_cs_diff_BV1G[(merged_df_cs_diff_BV1G['amb']=='Both')]
merged_df_cs_diff_BV1G = merged_df_cs_diff_BV1G[(merged_df_cs_diff_BV1G['Metab']=='GABA+')]


In [ ]:
# Drop rows with missing values
df_clean = merged_df_cs_diff_BV1G.dropna(subset=['metab_difference', 'ABC'])

# Calculate Pearson correlation
r, p = pearsonr(df_clean['metab_difference'], df_clean['ABC'])

# Plot
plt.figure(figsize=(8, 6))
sns.regplot(
    data=df_clean,
    x='ABC',
    y='metab_difference',
    ci=None,
    scatter_kws={'s': 60, 'alpha': 0.8}
)

# Annotate with correlation coefficient and p-value
plt.title(f'Contrast Sensitivity - V1 - GABA+ - Correlation: r = {r:.2f}, p = {p:.3f}', fontsize=14)
plt.xlabel('EI Difference')
plt.ylabel('Disparity Difference')
plt.grid(True)
plt.tight_layout()
plt.show()

# Interocular Imbalance
Correlation between metabolite change and the interocular-imbalance ABC measure.

In [ ]:
# Split the dataframe into pretraining and posttraining
pretraining = merged_df_bi[merged_df_bi['stage'] == 'pre'].copy()
posttraining = merged_df_bi[merged_df_bi['stage'] == 'post'].copy()

# Merge on common columns
merged = pd.merge(
    posttraining, 
    pretraining, 
    on=['ID', 'location', 'ABC', 'Metab'], 
    suffixes=('_post', '_pre')
)

# Calculate the differences
merged['metab_difference'] = merged['metab_concentration_post'] - merged['metab_concentration_pre']

# Create a reduced DataFrame with only relevant columns
merged_df_bi_diff = merged[['ID', 'location', 'Metab', 'metab_difference', 'ABC']]

# Display the resulting DataFrame
#print(merged_cutoff_diff)


plot_grouped_scatter(
    df=merged_df_bi_diff,
    group_by=[ 'location', 'Metab'],  # Grouping by 'amb' and 'location'
    cols=2,  # Number of columns in the subplot grid
    figsize=(15, 5),  # Base figure size
    x='ABC',  # X-axis column
    y='metab_difference',  # Y-axis column
    hue='ID',  # Hue for scatter plot
    style='ID',  # Style for scatter plot
    ax_title_format='{location} | {metab}',  # Dynamic title based on grouping keys
    xlabel='Area Below Curve',  # X-axis label
    ylabel='Metabolite Concentration Difference (post-pre)',  # Y-axis label
    save_path=None  # Show the plot instead of saving
)
#############################################################################################################
#############################################################################################################
